# Notebook 04: Operations & Monitoring

**Persona:** ML Engineer / Data Engineer

**Purpose:** Monitor and operate the Feature Store infrastructure:
1. Dynamic Table health & refresh monitoring
2. Feature View metadata and lineage
3. Impact analysis for upstream schema changes
4. Feature freshness monitoring
5. Operational queries for troubleshooting

**Prerequisites:** Run Notebooks 00, 01, 02, and 03 first

## 1. Setup

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore")

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except Exception:
    sys.path.insert(0, ".")
    from feature_definitions.config import get_session, ROLES
    session = get_session(role=ROLES["admin"])

sys.path.insert(0, ".") if "." not in sys.path else None

from feature_definitions.config import get_config, ROLES
from snowflake.ml.feature_store import FeatureStore, CreationMode
import pandas as pd

cfg = get_config("DEV")
session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

fs = FeatureStore(
    session=session,
    database=cfg["database"],
    name=cfg["fs_schema"],
    default_warehouse=cfg["warehouse"],
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)
print(f"Monitoring: {cfg['database']}.{cfg['fs_schema']}")

## 2. Dynamic Table Health

Dynamic Tables are the backbone of DT-based and tiled Feature Views. All 9 DTs should show `ACTIVE` scheduling state.

In [ ]:
# List all Dynamic Tables using SHOW command
dt_rows = session.sql(f"""
    SHOW DYNAMIC TABLES IN SCHEMA {cfg["database"]}.{cfg["fs_schema"]}
""").collect()

dt_data = []
for r in dt_rows:
    d = r.as_dict()
    dt_data.append({
        "Name": d["name"],
        "State": d.get("scheduling_state", "UNKNOWN"),
        "Target Lag": d.get("target_lag", "N/A"),
        "Refresh Mode": d.get("refresh_mode", "N/A"),
    })

dt_df = pd.DataFrame(dt_data)
print(f"Dynamic Tables: {len(dt_df)}")
print(f"Active: {len(dt_df[dt_df['State'] == 'ACTIVE'])}")
print(f"\n{dt_df.to_string(index=False)}")

## 3. Refresh History

Track DT refresh performance and identify failures. Each row represents one incremental or full refresh cycle.

In [ ]:
refresh = session.sql(f"""
    SELECT
        name AS NAME,
        state AS STATE,
        refresh_start_time AS START_TIME,
        DATEDIFF(second, refresh_start_time, refresh_end_time) AS DURATION_SEC,
        statistics:numInsertedRows::INT AS ROWS_INSERTED,
        statistics:numDeletedRows::INT  AS ROWS_DELETED
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
        NAME_PREFIX => '{cfg["database"]}.{cfg["fs_schema"]}.'
    ))
    WHERE refresh_start_time >= DATEADD(hour, -24, CURRENT_TIMESTAMP())
    ORDER BY refresh_start_time DESC
    LIMIT 50
""").collect()

if refresh:
    refresh_data = [r.as_dict() for r in refresh]
    ref_df = pd.DataFrame(refresh_data)
    print(f"Refreshes in last 24h: {len(ref_df)}")
    print(f"States: {dict(ref_df['STATE'].value_counts())}")
    print(f"\n{ref_df[['NAME','STATE','DURATION_SEC','ROWS_INSERTED']].head(15).to_string(index=False)}")
else:
    print("No refresh history in last 24 hours")

## 4. Feature View Inventory

Complete catalog of all registered Feature Views with type classification and entity mappings.

In [ ]:
fvs = fs.list_feature_views().to_pandas()
entities = fs.list_entities().to_pandas()

print(f"Feature Views: {len(fvs)}")
print(f"Entities: {len(entities)}")

# Classify FV types
def classify_fv(row):
    rf = row.get("REFRESH_FREQ")
    if pd.isna(rf) or rf == "None" or rf is None:
        return "View"
    return "DT/Tiled"

fvs["TYPE"] = fvs.apply(classify_fv, axis=1)
print(f"\n{fvs[['NAME','VERSION','TYPE','REFRESH_FREQ']].to_string(index=False)}")
print(f"\nBy type: {dict(fvs['TYPE'].value_counts())}")

## 5. Entity Inventory

In [ ]:
entities = fs.list_entities().to_pandas()
print(entities[["NAME", "JOIN_KEYS", "DESC"]].to_string(index=False))

## 6. Impact Analysis

Understand which Feature Views and downstream models would be affected by upstream changes.

In [ ]:
# Map source tables to Feature Views
print("=== Source Table → Feature View Mapping ===\n")

# We know the architecture from feature_definitions/
dependencies = {
    "SESSIONS": ["SESSION_BEHAVIOR_FEATURES", "USER_RECENCY_RAW",
                 "USER_PERMUTATION_FEATURES_WIDE"],
    "EVENTS":   ["USER_SESSION_ENGAGEMENT", "USER_ENGAGEMENT_REALTIME",
                 "USER_PERMUTATION_FEATURES_WIDE"],
    "ORDERS":   ["USER_PURCHASE_AGGREGATES", "USER_TREND_FEATURES",
                 "USER_ORDER_SUMMARY_DBT"],
    "USERS":    ["USER_PROFILE_FEATURES", "USER_RECENCY_RAW"],
    "PRODUCTS": ["PRODUCT_CATALOG_FEATURES", "PRODUCT_EMBEDDINGS"],
    "CATEGORIES": ["PRODUCT_CATALOG_FEATURES"],
    "PRODUCT_SUPPLIER": ["PRODUCT_SUPPLIER_METRICS"],
}

for table, fv_list in dependencies.items():
    print(f"  {table:20s} → {', '.join(fv_list)}")

print("\n--- Simulated Impact: SESSIONS table schema change ---")
affected_fvs = dependencies.get("SESSIONS", [])
print(f"Directly affected FVs: {affected_fvs}")
print(f"Downstream models to review: CONVERSION_PREDICTION (uses SESSION_BEHAVIOR_FEATURES)")

## 7. Operational Queries

In [ ]:
# 7.1 Warehouse credit usage
print("=== Warehouse Credits (last 7 days) ===")
try:
    credits = session.sql(f"""
        SELECT
            DATE_TRUNC('day', START_TIME)::DATE AS DAY,
            SUM(CREDITS_USED) AS CREDITS
        FROM SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY
        WHERE WAREHOUSE_NAME = '{cfg["warehouse"]}'
          AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
        GROUP BY 1 ORDER BY 1 DESC
    """).collect()
    if credits:
        for r in credits:
            d = r.as_dict()
            print(f"  {d['DAY']}  {d['CREDITS']:.4f} credits")
    else:
        print(f"  No usage for {cfg['warehouse']}")
except Exception as e:
    print(f"  Cannot query Account Usage: {e}")

In [ ]:
# 7.2 Model registry summary
print("=== Model Registry ===")
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name=cfg["database"],
    schema_name=cfg["ml_datasets_schema"],
)
models = registry.show_models()
for _, m in models.iterrows():
    model = registry.get_model(m["name"])
    versions = model.show_versions()
    print(f"  {m['name']}: {len(versions)} version(s)")
    for _, v in versions.iterrows():
        print(f"    {v['name']} - created {v['created_on']}")

In [ ]:
# 7.3 Dataset versions
print("=== Training Datasets ===")
try:
    ds_results = session.sql(f"""
        SHOW DATASETS IN SCHEMA {cfg["database"]}.{cfg["fs_schema"]}
    """).collect()
    if ds_results:
        for r in ds_results:
            d = r.as_dict()
            print(f"  {d.get('name', 'unknown')}  created={d.get('created_on', '?')}")
    else:
        print("  No datasets found")
except Exception as e:
    print(f"  {e}")

## 8. Pipeline Latency Monitoring

If ROW_TIMESTAMP is enabled (see Notebooks 00 and 05), we can measure data freshness and pipeline latency using `METADATA$ROW_LAST_COMMIT_TIME`.

In [ ]:
from feature_definitions.latency import (
    get_source_freshness,
    get_dt_freshness,
    measure_stage_latency,
    get_batch_latency,
    pipeline_summary,
)

# Source table freshness
print("=== Source Table Freshness ===")
source_fresh = get_source_freshness(session, "DEV")
for s in source_fresh:
    age = s.get("AGE_SECONDS")
    if age is not None:
        print(f"  {s['TABLE']:20s} {age:>6}s old")
    else:
        print(f"  {s['TABLE']:20s} ROW_TIMESTAMP not enabled or no data")

# DT freshness
print("\n=== DT Feature View Freshness ===")
session.sql(f"USE ROLE {ROLES['dev']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()
dt_fresh = get_dt_freshness(session, "DEV")
for d in dt_fresh:
    age = d.get("AGE_SECONDS")
    if age is not None:
        print(f"  {d['TABLE']:45s} {age:>6}s old")
    else:
        print(f"  {d['TABLE']:45s} awaiting refresh or not enabled")
session.sql(f"USE ROLE {ROLES['admin']}").collect()
session.sql(f"USE WAREHOUSE {cfg['warehouse']}").collect()

# Stage latency
print("\n=== Pipeline Stage Latency ===")
stages = measure_stage_latency(session, "DEV")
for s in stages:
    lat = s.get("LATENCY_SECONDS")
    status = f"{lat}s" if lat is not None else "N/A"
    print(f"  {s['FROM_TABLE']:20s} → {s['TO_TABLE']:40s} {status}")

# Generator batch log (if available)
print("\n=== Recent Generator Batches ===")
try:
    batches = get_batch_latency(session, "DEV", last_n=5)
    for b in batches:
        print(f"  Batch {b.get('LOG_ID','?')}: "
              f"{b.get('SESSIONS_GENERATED','?')} sessions, "
              f"duration={b.get('DURATION_MS','?')}ms")
except Exception as e:
    print(f"  Generator not deployed yet ({e})")

## Summary

This notebook provides the operational toolkit for managing a Feature Store deployment:

| Capability | Method |
|---|---|
| DT health | `SHOW DYNAMIC TABLES` |
| Refresh history | `INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY()` |
| Feature inventory | `fs.list_feature_views()` + `fs.list_entities()` |
| Impact analysis | Dependency mapping from `feature_definitions/` |
| Credit monitoring | `SNOWFLAKE.ACCOUNT_USAGE.WAREHOUSE_METERING_HISTORY` |
| Model tracking | Model Registry API |
| Pipeline latency | `METADATA$ROW_LAST_COMMIT_TIME` via `feature_definitions/latency.py` |

For an interactive dashboard, see `monitoring_dashboard/streamlit_app.py`.

For detailed pipeline performance testing and latency visualisation, see **Notebook 05: Pipeline Performance**.